# QLoRA Training on Google Colab

This notebook is designed to fine-tune a Vision-Language Model (like Florence-2) using QLoRA on a free T4 GPU.

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets
!pip install -q flash-attn --no-build-isolation

In [ ]:
# Mount Google Drive to load datasets
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "microsoft/Florence-2-base"
print(f"Loading {model_id} in 4-bit precision...")

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    device_map="auto",
    load_in_4bit=True
)

model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

In [ ]:
print("Loading dataset from Google Drive...")
# Update this path to point to where you uploaded qlora_train.jsonl in your drive
dataset_path = "/content/drive/MyDrive/datasets/qlora_train.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")

training_args = TrainingArguments(
    output_dir="./qlora-florence2-address",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    optim="paged_adamw_8bit",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()
model.save_pretrained("/content/drive/MyDrive/final_qlora_adapter")
print("Training Complete!")